## Preparación base de datos

In [1]:
pip install pandas matplotlib seaborn pyarrow

Note: you may need to restart the kernel to use updated packages.


#### Zonas de manhattan

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

manhattan_ids = [12, 13, 24, 41, 42, 43, 45, 48, 50, 68, 74, 75, 79,
                 87, 88, 90, 100, 103, 104, 105, 106, 107, 113, 114,
                 116, 120, 125, 127, 128, 137, 140, 141, 142, 143,
                 144, 148, 151, 152, 153, 158, 161, 162, 163, 164,
                 166, 170, 186, 202, 209, 211, 224, 229, 230, 231,
                 232, 233, 234, 236, 237, 239, 243, 244, 246, 249,
                 261, 262, 263]

#print(df.head())

#### Filtrar y unir base de datos del 2024 y 2025 de taxi amarillos:

In [3]:
import os

In [4]:
import glob

ruta_taxi = "/Users/jmatas/OneDrive/Universidad/8vo semestre/Capstone/Datos taxi amarillo/2024"
meses_otoño_2024 = ["09","10","11","12"]
archivos_taxi_2024 = [os.path.join(ruta_taxi, f"yellow_tripdata_2024-{m}.parquet")
                      for m in meses_otoño_2024]
dfs = []
for archivo in archivos_taxi_2024:
    df_temp = pd.read_parquet(archivo)
    df_temp = df_temp[
        (df_temp["PULocationID"].isin(manhattan_ids)) & 
        (df_temp["DOLocationID"].isin(manhattan_ids))
    ].copy()
    df_temp['tpep_pickup_datetime'] = pd.to_datetime(df_temp['tpep_pickup_datetime'])
    df_temp['month'] = df_temp['tpep_pickup_datetime'].dt.month
    df_temp['weekday'] = df_temp['tpep_pickup_datetime'].dt.day_name()
    dfs.append(df_temp)

df_all_taxi_2024 = pd.concat(dfs, ignore_index=True)
df_all_taxi_2024 = df_all_taxi_2024[df_all_taxi_2024['month'].between(9, 12)]
demanda_2024_taxi = df_all_taxi_2024.groupby(['month', 'weekday']).size().reset_index(name='count')
viajes_por_mes_taxi_2024 = df_all_taxi_2024['month'].value_counts().sort_index()
print(viajes_por_mes_taxi_2024)

month
9     2828630
10    3000441
11    2879470
12    2882393
Name: count, dtype: int64


In [5]:
ruta_taxi = "/Users/jmatas/OneDrive/Universidad/8vo semestre/Capstone/Datos taxi amarillo/2025"
archivos_taxi_2025 = sorted(glob.glob(f"{ruta_taxi}/yellow_tripdata_2025-0[1-6].parquet"))
dfs = []
for archivo in archivos_taxi_2025:
    df_temp = pd.read_parquet(archivo)
    df_temp = df_temp[
        (df_temp["PULocationID"].isin(manhattan_ids)) & 
        (df_temp["DOLocationID"].isin(manhattan_ids))
    ].copy()
    df_temp['tpep_pickup_datetime'] = pd.to_datetime(df_temp['tpep_pickup_datetime'])
    df_temp['month'] = df_temp['tpep_pickup_datetime'].dt.month
    df_temp['weekday'] = df_temp['tpep_pickup_datetime'].dt.day_name()
    dfs.append(df_temp)

df_all_taxi_2025 = pd.concat(dfs, ignore_index=True)
df_all_taxi_2025 = df_all_taxi_2025[df_all_taxi_2025['month'].between(1, 6)]
demanda_2025_taxi = df_all_taxi_2025.groupby(['month', 'weekday']).size().reset_index(name='count')
viajes_por_mes_taxi_2025 = df_all_taxi_2025['month'].value_counts().sort_index()
print(viajes_por_mes_taxi_2025)

month
1    2776626
2    2845540
3    3204159
4    3071232
5    3464634
6    3232329
Name: count, dtype: int64


In [6]:
# Agregar columna de año a cada base
df_all_taxi_2024['year'] = 2024
df_all_taxi_2025['year'] = 2025
# Unir bases
df_all_taxi = pd.concat([df_all_taxi_2024, df_all_taxi_2025], ignore_index=True)
#print("Tamaño total:", len(df_all_taxi))
#print("Por año:\n", df_all_taxi['year'].value_counts().sort_index())
print("Por mes:\n", df_all_taxi.groupby(['year','month']).size())

Por mes:
 year  month
2024  9        2828630
      10       3000441
      11       2879470
      12       2882393
2025  1        2776626
      2        2845540
      3        3204159
      4        3071232
      5        3464634
      6        3232329
dtype: int64


In [7]:
# Renombrar columnas clave para igualarlas con la de FHV
df_all_taxi = df_all_taxi.rename(columns={
    "tpep_pickup_datetime": "pickup_datetime",
    "tpep_dropoff_datetime": "dropOff_datetime",
    "PULocationID": "PUlocationID",
    "DOLocationID": "DOlocationID"
})
# Seleccionar solo las columnas que nos interesan
df_all_taxi = df_all_taxi[[
    "pickup_datetime", "dropOff_datetime", "PUlocationID", "DOlocationID", "month", "weekday", "year"
]]

#### Filtrar y unir base de datos de 2024 y 2025 de FHV:

In [8]:
# Cargar y unir los 6 archivos de meses
archivos = sorted(glob.glob("Datos/fhv_tripdata_2025-0[1-6].parquet"))
dfs = []
for archivo in archivos:
    df_temp = pd.read_parquet(archivo)
    df_temp = df_temp[
        (df_temp["PUlocationID"].isin(manhattan_ids)) & 
        (df_temp["DOlocationID"].isin(manhattan_ids))
    ].copy()
    df_temp['pickup_datetime'] = pd.to_datetime(df_temp['pickup_datetime'])
    df_temp['month'] = df_temp['pickup_datetime'].dt.month
    df_temp['weekday'] = df_temp['pickup_datetime'].dt.day_name()
    dfs.append(df_temp)
df_all = pd.concat(dfs, ignore_index=True)

In [9]:
ruta = "/Users/jmatas/OneDrive/Universidad/8vo semestre/Capstone/Datos 2024"
archivos_2024 = sorted(glob.glob(f"{ruta}/fhv_tripdata_2024-*.parquet"))
dfs = []
for archivo in archivos_2024:
    df_temp = pd.read_parquet(archivo)
    df_temp = df_temp[
        (df_temp["PUlocationID"].isin(manhattan_ids)) & 
        (df_temp["DOlocationID"].isin(manhattan_ids))
    ].copy()
    df_temp['pickup_datetime'] = pd.to_datetime(df_temp['pickup_datetime'])
    df_temp['month'] = df_temp['pickup_datetime'].dt.month
    df_temp['weekday'] = df_temp['pickup_datetime'].dt.day_name()
    dfs.append(df_temp)
df_all_2024 = pd.concat(dfs, ignore_index=True)

In [10]:
# Limpiar y agregar columna year
fhv_2024_clean = df_all_2024[['pickup_datetime','dropOff_datetime',
                              'PUlocationID','DOlocationID','month','weekday']].copy()
fhv_2024_clean['year'] = fhv_2024_clean['pickup_datetime'].dt.year
fhv_2024_clean = fhv_2024_clean[fhv_2024_clean['month'].between(9,12)]

fhv_2025_clean = df_all[['pickup_datetime','dropOff_datetime',
                         'PUlocationID','DOlocationID','month','weekday']].copy()
fhv_2025_clean['year'] = fhv_2025_clean['pickup_datetime'].dt.year
fhv_2025_clean = fhv_2025_clean[fhv_2025_clean['month'].between(1,6)]

# Concatenar en uno solo
fhv_all = pd.concat([fhv_2024_clean, fhv_2025_clean], ignore_index=True)

# Chequeo rápido
#print(fhv_all.info())
print(fhv_all[['year','month']].value_counts().sort_index())

year  month
2024  9        51152
      10       49418
      11       54238
      12       12130
2025  1        51450
      2        44205
      3        15195
      4        42652
      5        48827
      6        46659
Name: count, dtype: int64


#### Unimos la bd de fhv con taxi amarillos

In [11]:
# 1) FHV 
fhv_all['tipo'] = 'fhv'
# 2) Taxi amarillo
df_all_taxi['tipo'] = 'taxi'
# 3) Unir ambas bases
df_all = pd.concat([fhv_all, df_all_taxi], ignore_index=True)

# 4) Chequeo rápido
print(df_all['tipo'].value_counts())
print(df_all[['year','month','tipo']].value_counts().sort_index())

tipo
taxi    30185454
fhv       415926
Name: count, dtype: int64
year  month  tipo
2024  9      fhv       51152
             taxi    2828630
      10     fhv       49418
             taxi    3000441
      11     fhv       54238
             taxi    2879470
      12     fhv       12130
             taxi    2882393
2025  1      fhv       51450
             taxi    2776626
      2      fhv       44205
             taxi    2845540
      3      fhv       15195
             taxi    3204159
      4      fhv       42652
             taxi    3071232
      5      fhv       48827
             taxi    3464634
      6      fhv       46659
             taxi    3232329
Name: count, dtype: int64


#### df_all es la base de datos con la que se trabajará

In [13]:
# Filtrar df_all solo para primavera y otoño (calendario meteorológico)
df_all = df_all[df_all["month"].isin([3,4,5,9,10,11])].copy()
print("Número de filas después del filtro:", len(df_all))

Número de filas después del filtro: 18710048


----------------------------------------------
----------------------------------------------
<br><br>